In [16]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import inequality_analyzers as inqA
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise

In [17]:
# Load final data
df_main = pd.read_csv(data_root / 'CBOS_data_final.csv')

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_65042/2647765996.py:2: DtypeWarning: Columns (14,15,20,21,22,38,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df_main = pd.read_csv(data_root / 'CBOS_data_final.csv')


In [18]:
def extract_income_categorized(df):
    import re
    # Pattern for range like "601 - 900 tys. zł." or "1.501 do 2.500 tys. zł" or with en-dash "1.451 – 1.700 tys. zł"
    pattern = r'^\s*(\d+(?:\.\d+)?)\s*(?:-|–|do)\s*(\d+(?:\.\d+)?)\s*(?:tys\.?)?\s*(?:zł\.?)?\s*$'
    
    def parse_income(value):
        if pd.isna(value):
            return [0, 0]
        
        value_str = str(value).strip()
        
        # Skip missing data markers
        if value_str in ['BRAK DANYCH / Odmowa odpowiedzi', 'BRAK DANYCH', 'Odmowa odpowiedzi']:
            return [0, 0]
        
        # Try to match range pattern first
        match = re.match(pattern, value_str)
        if match:
            low = float(match.group(1))
            high = float(match.group(2))
            # If values are less than 10 with decimal notation, multiply by 1000 (they're in thousands)
            if low < 10 and '.' in match.group(1):
                low *= 1000
            if high < 10 and '.' in match.group(2):
                high *= 1000
            return [((low + high) / 2) * 1000, 2]
        
        # Try to match "Nie/nie więcej niż X" or "Nie/nie mniej niż X" patterns (case-insensitive)
        match_nie = re.match(r'^\s*[Nn]ie\s+(?:więcej|mniej)\s+niż\s+(\d+(?:\.\d+)?)\s*(?:tys\.?)?\s*(?:zł\.?)?\s*$', value_str)
        if match_nie:
            single_value = float(match_nie.group(1))
            return [single_value * 1000, 1]
        
        # Try to match "powyżej X" (more than) or "do X" (up to) patterns
        match_limit = re.match(r'^\s*(?:powyżej|do)\s+(\d+(?:\.\d+)?)\s*(?:tys\.?)?\s*(?:zł\.?)?\s*$', value_str)
        if match_limit:
            single_value = float(match_limit.group(1))
            return [single_value * 1000, 1]
        
        # Try to match "X tys. zł i mniej" or "X tys. zł i więcej" (number first, then tys/zł, then qualifier)
        match_qualifier = re.match(r'^\s*(\d+(?:\.\d+)?)\s*(?:tys\.?)?\s*(?:zł\.?)?\s*(?:i|lub)\s*(?:więcej|mniej)\s*(?:tys\.?)?\s*(?:zł\.?)?\s*$', value_str)
        if match_qualifier:
            single_value = float(match_qualifier.group(1))
            return [single_value * 1000, 1]
        
        # Try to match "X i więcej tys. zł." (number, qualifier, then tys/zł)
        match_qualifier_alt = re.match(r'^\s*(\d+(?:\.\d+)?)\s*(?:i|lub)\s*(?:więcej|mniej)\s*(?:tys\.?)?\s*(?:zł\.?)?\s*$', value_str)
        if match_qualifier_alt:
            single_value = float(match_qualifier_alt.group(1))
            return [single_value * 1000, 1]
        
        # If no match found, print for debugging
        print(f"No match: {value_str}")
        return [0, 0]
        
    values = []
    codes = []
    for row in df.itertuples():
        income_value = row.income_p_T_L
        parser = parse_income(income_value)
        parsed_value = parser[0] if parser[0] > 0 else np.nan
        code = parser[1]
        values.append(parsed_value)
        codes.append(code)
    
    values = pd.Series(values, index=df.index)
    codes = pd.Series(codes, index=df.index)
 
    top_or_bottom_value = values[codes == 1]
    if len(top_or_bottom_value) > 0:
        top = ((top_or_bottom_value.max()) + (9991 * 1000)) * 0.5
        bottom = (top_or_bottom_value.min() * 0.5)
        
        values[(codes == 1) & (values == top_or_bottom_value.max())] = top
        values[(codes == 1) & (values == top_or_bottom_value.min())] = bottom
    
    df = df.copy()
    df['income_hh'] = values / 10000
    mean_hh_size = np.mean(df[~df['household_size'].isna()]['household_size'].values)
    df['income_hh'] = df['income_hh'] / mean_hh_size
    return df

# Process each survey file separately
for file in df_main['survey file'].unique():
    mask = (df_main['survey file'] == file)
    df_subset = df_main.loc[mask].copy()
    if df_subset['income_hh'].sum() == 0.0:
        print(f"Processing survey file: {file}")
        df_processed = extract_income_categorized(df_subset)
        df_main.loc[mask, 'income_hh'] = df_processed['income_hh']


Processing survey file: CBOS_7_07_1990.sav
Processing survey file: CBOS_8_09_1990.sav
Processing survey file: CBOS_9_10_1990.sav
Processing survey file: CBOS_10_11_1990.sav
Processing survey file: CBOS_11_12_1990.sav
Processing survey file: CBOS_12_01_1991.sav
Processing survey file: CBOS_13_02_1991.sav
Processing survey file: CBOS_14_03_1991.sav
Processing survey file: CBOS_15_04_1991.sav
Processing survey file: CBOS_16_05_1991.sav
Processing survey file: CBOS_17_06_1991.sav
Processing survey file: CBOS_18_07_1991.sav
Processing survey file: CBOS_19_08_1991.sav
Processing survey file: CBOS_20_09_1991.sav
Processing survey file: CBOS_21_10_1991.sav
Processing survey file: CBOS_22_11_1991.sav
Processing survey file: CBOS_23_01_1992.sav


In [19]:
t = df_main[df_main['income_p_T_L'].notna()][['income_hh', 'income_p_T_L', 'survey file']]
t


,income_hh,income_p_T_L,survey file
8837,20.610454,601 - 900 tys. zł.,CBOS_7_07_1990.sav
8838,12.371765,301 - 600 tys. zł.,CBOS_7_07_1990.sav
8839,12.371765,301 - 600 tys. zł.,CBOS_7_07_1990.sav
8840,28.849144,901 - 1.200 tys. zł.,CBOS_7_07_1990.sav
8841,28.849144,901 - 1.200 tys. zł.,CBOS_7_07_1990.sav
...,...,...,...
30671,30.742966,751 - 1.500 tys. zł,CBOS_23_01_1992.sav
30672,30.742966,751 - 1.500 tys. zł,CBOS_23_01_1992.sav
30673,54.643540,1.501 do 2.500 tys. zł,CBOS_23_01_1992.sav
30674,30.742966,751 - 1.500 tys. zł,CBOS_23_01_1992.sav


In [20]:
unmatched = t[t['income_p_T_L'].notna() & (t['income_hh'].isna())][['income_hh', 'income_p_T_L', 'survey file']]
print(f"Total non-null income_p_T_L entries: {len(t)}")
print(f"Successfully parsed (with income_hh values): {len(t[t['income_hh'].notna()])}")
print(f"Unmatched entries: {len(unmatched)}")
print("\nUnique unmatched values:")
for val in unmatched['income_p_T_L'].unique()[:20]:
    print(f"  '{val}'")
unmatched.head(20)


Total non-null income_p_T_L entries: 21695
Successfully parsed (with income_hh values): 21686
Unmatched entries: 9

Unique unmatched values:
  'BRAK DANYCH / Odmowa odpowiedzi'


,income_hh,income_p_T_L,survey file
29710,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
29711,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30102,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30103,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30252,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30402,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30445,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30447,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav
30552,NaN,BRAK DANYCH / Odmowa odpowiedzi,CBOS_23_01_1992.sav


In [21]:
df_main.to_csv(data_root / 'CBOS_data_final_with_income.csv', index=False)